This notebook is for modeling and evaluating span identification from the SemEval dataset. This is the first filter for propaganda, so we only want to filter out what we are confident is not propaganda (so high-sensitivity/high-recall). Then downstream, let the technique classification (TC) model handle the precision and pruning.

In [1]:
import os
import json
from torch import nn, torch
import pandas as pd
import numpy as np
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
from sklearn.metrics import precision_recall_fscore_support
from datasets import Dataset
import evaluate
import tqdm
from tqdm.auto import tqdm as tqdm_auto
tqdm_auto.pandas()
from accelerate.state import AcceleratorState
from transformers.utils.notebook import NotebookProgressCallback
AcceleratorState._reset_state()

In [2]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [3]:
#Set up paths
BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data" / "processed"
MODEL_DIR = BASE_DIR / "models" / "semeval_roberta_scanner"

In [4]:
#Check for GPU support
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple Metal (MPS) for acceleration")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using NVIDIA GPU")
else:
    device = torch.device("cpu")
    print("Using CPU. Training might be slow.")

Using Apple Metal (MPS) for acceleration


In [5]:
#Load article-level span identification data
df_si = pd.read_csv(DATA_DIR / "semeval_si_cleaned.csv")
df_si['propaganda_offsets'] = df_si['propaganda_offsets'].apply(json.loads)
print(f"Loaded {len(df_si)} articles.")
df_si.head()

Loaded 357 articles.


,article_id,text,propaganda_offsets
0,111111111,Next plague outbreak in Madagascar could be 's...,"[[265, 323], [1795, 1935], [149, 157], [1069, ..."
1,111111112,US bloggers banned from entering UK\n\nTwo pro...,"[[191, 219], [476, 556], [785, 798], [958, 101..."
2,111111113,Kate Steinle's death at the hands of a Mexican...,"[[1396, 1430], [3082, 3099], [3828, 3985], [36..."
3,111111114,U.S. judge frees Indonesian immigrant held by ...,"[[1705, 1824]]"
4,111111115,Here are all the sexual misconduct accusations...,"[[658, 700], [1870, 1893], [1655, 1745], [2389..."


In [6]:
#Initialize the model
##Tried "bert-base-uncased", Best F1 score after 3 epochs: 0.26392
##"microsoft/deberta-v3-small", After 3 epoches, still getting gradient explosion every time, even after adjusting hyperparameters
model_checkpoint = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, add_prefix_space=True)

In [7]:
#Initialize model - Load from local if exists, else from checkpoint
if (MODEL_DIR / "config.json").exists():
    print(f"Loading existing trained model from: {MODEL_DIR}")
    model = AutoModelForTokenClassification.from_pretrained(MODEL_DIR)
    model_already_trained = True
else:
    print(f"No existing model found. Initializing from: {model_checkpoint}")
    model = AutoModelForTokenClassification.from_pretrained(model_checkpoint, num_labels=3)
    model_already_trained = False

model.to(device)


Loading existing trained model from: ../models/semeval_roberta_scanner


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForTokenClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (L

In [8]:
def tokenize_and_align_labels(examples):
    # 1. Tokenize with a sliding window
    tokenized_inputs = tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    # 2. Correctly map chunks to original articles
    # 'overflow_to_sample_mapping' is the correct key for Fast Tokenizers
    sample_mapping = tokenized_inputs.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized_inputs.pop("offset_mapping")

    labels = []

    # Iterate through every chunk (i) and find which original article (sample_idx) it belongs to
    for i, offsets in enumerate(offset_mapping):
        sample_idx = sample_mapping[i]
        article_spans = examples["propaganda_offsets"][sample_idx]

        doc_labels = []
        for start, end in offsets:
            # -100 for special tokens like [CLS], [SEP], and [PAD]
            if start == end == 0:
                doc_labels.append(-100)
                continue

            # Check if this specific token character-range overlaps with any propaganda span
            is_prop = any(s <= start < e or s < end <= e for s, e in article_spans)

            # Label as 1 (Propaganda) or 0 (Normal)
            doc_labels.append(1 if is_prop else 0)

        labels.append(doc_labels)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [9]:
#Tokenize and train-test split dataset
raw_dataset = Dataset.from_pandas(df_si).train_test_split(test_size=0.2)
tokenized_datasets = raw_dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_dataset["train"].column_names
)

Map:   0%|          | 0/285 [00:00<?, ? examples/s]

Map:   0%|          | 0/72 [00:00<?, ? examples/s]

In [10]:
#Evaluate model
metric = evaluate.load("seqeval")
label_list = ["O", "B-Prop", "I-Prop"]

In [11]:
def compute_metrics(p):
    predictions, labels = p
    #Get the highest probability label for each token
    predictions = np.argmax(predictions, axis=2)

    #Flatten the lists and remove the -100 (ignore) tokens
    y_true = labels.flatten()
    y_pred = predictions.flatten()

    #Filter out -100 labels so they don't count towards accuracy
    mask = y_true != -100
    y_true = y_true[mask]
    y_pred = y_pred[mask]

    #Calculate token-level precision, recall, and F1
    #'pos_label=1' tells it that label 1 is the "Propaganda" we care about
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='binary', pos_label=1
    )

    #Calculate F2 (Weights Recall 2x more than Precision)
    f2 = (5 * precision * recall) / (4 * precision + recall) if (4 * precision + recall) > 0 else 0

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "f2_score": f2
    }

In [12]:
#Tried without weighting before and was quickly overfitting
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Prioritize Recall: Propaganda classes (1, 2) weighted 20x more than background (0)
        weights = torch.tensor([1.0, 20.0, 20.0], device=model.device)
        loss_fct = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

In [13]:
#Set up training arguments with optimized hyperparameters
training_args = TrainingArguments(
    output_dir=MODEL_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2.558e-05,
    per_device_train_batch_size=4,
    num_train_epochs=10,
    weight_decay=0.0817,
    logging_steps=5,
    metric_for_best_model="f2_score",
    dataloader_pin_memory=False,
    disable_tqdm=False,
    report_to="none",
    load_best_model_at_end=True
)

In [14]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics
)

In [15]:
#Train only if we didn't load a local model
if not model_already_trained:
    print("Starting training process...")
    trainer.train()
    trainer.save_model(MODEL_DIR)
    print(f"Model trained and saved to {MODEL_DIR}")
else:
    print("Model was already loaded from disk. Skipping training.")
    trainer.remove_callback(NotebookProgressCallback)

Model was already loaded from disk. Skipping training.


In [16]:
#Evaluate performance on the test dataset
test_results = trainer.evaluate(eval_dataset=tokenized_datasets["test"])

print("\n" + "="*30)
print("FINAL MODEL PERFORMANCE")
print(f"Recall:    {test_results['eval_recall']:.4f}")
print(f"Precision: {test_results['eval_precision']:.4f}")
print(f"F2 Score:  {test_results['eval_f2_score']:.4f}")
print("="*30)


FINAL MODEL PERFORMANCE
Recall:    0.9613
Precision: 0.3841
F2 Score:  0.7391


In [17]:
#Compare results on train vs. test sets to ensure not overfitting
# Force evaluation on the Train set
train_results = trainer.evaluate(eval_dataset=tokenized_datasets["train"])
print(f"TRAIN Recall: {train_results['eval_recall']:.4f}")


TRAIN Recall: 0.9533
